# GNSS Constellation Visibility & Link Budget Analysis

This notebook determines which GNSS satellites are visible, given a user's reference trajectory.

In [ ]:
import numpy as np
from pathlib import Path
from orbit_determination.gnss_visibility import VisibilitySimulator

## 1. Simulation Configuration
Import the receiver's reference trajectory and attitude, define the receiver antenna orientation in the body frame, set the maximum reception and transmission off-boresight angles, and choose the minimum $C/N_0$ threshold for signal acquisition.

In [ ]:
# Path to the specific receiver trajectory/attitude file
project_root = Path.cwd().parent
# Sample data corresponding to a satellite electric orbit raising
rcver_file = project_root / 'data' / 'raw' / 'receiver_orbits' / 'eor' / 'sample2.txt'
# Columns: JD, x[km], y[km], z[km], roll[rad], pitch[rad], yaw[rad]
rcver_data = np.loadtxt(rcver_file, skiprows=1, usecols=(0, 13, 14, 15, 25, 26, 27)).T
rcver_time_julian = rcver_data[0]
rcver_pos = 1e3 * np.column_stack((rcver_data[1], rcver_data[2], rcver_data[3])) # Convert km to m
rcver_roll = rcver_data[4]
rcver_pitch = rcver_data[5]
rcver_yaw = rcver_data[6]

# Path to the export directory
export_base_path = project_root / 'data' / 'processed' / 'visibility' / 'sample2'

# Antenna pointing vector in the satellite's body frame
rcver_antenna_dir_body = [0, 1, 0]  
# Receiver antenna max off-boresight angle 
rcver_max_offb_deg = 80  
# Transmitter antenna max off-boresight angle  
sat_max_offb_deg = 60               
# Minimum C/N0 for signal acquisition (dB-Hz)
min_ctnr = 20                       

# Initialize the simulator
simulator = VisibilitySimulator(
    export_base_path=export_base_path,
    rcver_time_julian=rcver_time_julian,
    rcver_pos=rcver_pos,
    rcver_roll=rcver_roll,
    rcver_pitch=rcver_pitch,
    rcver_yaw=rcver_yaw,
    rcver_antenna_dir_body=rcver_antenna_dir_body,
    rcver_max_offb_deg=rcver_max_offb_deg,
    sat_max_offb_deg=sat_max_offb_deg,
    min_ctnr=min_ctnr,
    traj_source_name=rcver_file.name
)

print("Visibility Simulator initialized successfully.")

## 2. Execute Visibility Pipeline
Calculate and export the satellite visibility data, Tx and Rx angles, and C/N0 values for each constellation. The outputs will automatically be saved to `data/processed/visibility/`.

In [ ]:
# Run the pipeline for all constellations
simulator.run_gps()
simulator.run_galileo()
simulator.run_glonass()
simulator.run_beidou()

print("\nAll visibility data calculated and exported.")